# Gemini Human Recognize

In [1]:
%pip install --upgrade --quiet google-genai pillow dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [3]:
PROJECT_ID = os.environ['PROJECT_ID']
LOCATION = os.environ['LOCATION']
GS_BUCKET = os.environ['GS_BUCKET']
BUCKET = os.environ['BUCKET']

In [4]:
MODEL_ID= "gemini-2.0-flash-001"

In [46]:
from google import genai
from google.genai import types
from google.genai.types import (
    GenerateContentConfig,
    GoogleSearch,
    Part,
    Tool,
)
from IPython.display import HTML, Markdown, display
import pandas as pd

In [6]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

In [103]:
def detect_with_gemini(image_path):
    #system_instruction="You are an image analysis AI. You can identify people in images and provide their names"
    system_instruction="당신은 이미지 분석하는 일을 담당합니다. 주어진 이미지에서 인물을 찾아서 인물의 이름을 알려주세요. 인물이 누구인지 모르거나, 100% 신뢰할 수 없다면 '모름'으로 답하세요."

    google_search_tool = Tool(google_search=GoogleSearch())

    grounding_config = GenerateContentConfig(
        temperature = 0.1,
        top_p = 0.99,
        max_output_tokens = 8192,
        response_modalities = ["TEXT"],
        system_instruction=system_instruction,
        tools=[google_search_tool]   
    )

    #prompt = f"""If you can identify the people in the image, provide their names ONLY. If you don't know them, Don't make up names and just respond with 'unknown'"""
    prompt = f"""설명을 붙이지 말고, 이름만 대답하세요."""

    with open(image_path, "rb") as f:
        image = f.read()

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[Part.from_bytes(data=image, mime_type="image/png"),
            prompt,],
        config=grounding_config
    )

    print(response.text)


In [58]:
def detect_with_gemini_url(image_path, percent, grounding=True):
    #system_instruction="You are an image analysis AI. You can identify people in images and provide their names"
    system_instruction=f"당신은 이미지 분석하는 일을 담당합니다. 주어진 이미지에서 인물을 찾아서 인물의 이름을 알려주세요. 인물이 누구인지 모르거나, {percent}% 신뢰할 수 없다면 '모름'으로 답하세요."

    google_search_tool = Tool(google_search=GoogleSearch())

    grounding_config = GenerateContentConfig(
        temperature = 0.1,
        top_p = 0.99,
        max_output_tokens = 8192,
        response_modalities = ["TEXT"],
        system_instruction=system_instruction,
        tools=[google_search_tool]   
    )

    if grounding:
        grounding_config = GenerateContentConfig(
            temperature = 0.1,
            top_p = 0.99,
            max_output_tokens = 8192,
            response_modalities = ["TEXT"],
            system_instruction=system_instruction,
            tools=[google_search_tool]   
        )
    else:
        grounding_config = GenerateContentConfig(
            temperature = 0.1,
            top_p = 0.99,
            max_output_tokens = 8192,
            response_modalities = ["TEXT"],
            system_instruction=system_instruction,  
        )

    #prompt = f"""If you can identify the people in the image, provide their names ONLY. If you don't know them, Don't make up names and just respond with 'unknown'"""
    prompt = f"""설명을 붙이지 말고, 이름만 대답하세요. 
                output : 홍길동 """


    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[Part.from_uri(file_uri=f"{GS_BUCKET}/human_images_4/{image_path}", mime_type="image/jpeg"),
            prompt,],
        config=grounding_config
    )

    print(response.text)
    return response.text

In [52]:
def evaluate(df):
    answer_df = pd.read_csv("./human_names.csv", index_col=False)
    df = df.sort_values(by="idx").reset_index(drop=True)
    df['answer'] = answer_df.iloc[:len(df), 0].values 
    df['name'] = df['name'].str.replace('\n', '', regex=False)

    comparison_result = df['name'] == df['answer']
    matching_count = comparison_result.sum()
    print(matching_count)
    total_count = len(df)
    unknown_count = (df['name'] == '모름').sum()

    print(f"Total :{total_count}, 모름 : {unknown_count}, 불일치 : {total_count - matching_count}, 모름 제외 불일치 : {total_count - matching_count - unknown_count}")
    return df

In [ ]:
def display_dataframe_with_images(df):

    html = df.copy()
    html = html.to_html(escape=False, index=False)

    display(HTML(html))

In [ ]:
df = pd.DataFrame(columns=['idx','image', 'name'])

for i in range(1,101):
    name = detect_with_gemini_url(f"image_{i}.jpeg", 100)    
    image_data = f"<img src=\"https://storage.googleapis.com/{BUCKET}/human_images_4/image_{i}.jpeg\" width=\"40%\"/>"
    new_row = pd.DataFrame({'idx': [i],'image': [image_data], 'name': [name.replace('\n', '')]})
    df = pd.concat([df, new_row], ignore_index=True)


뷔

모름
김호영

모름

김수현

모름

모름

모름

유관순

아이유

모름

손흥민

모름
모름

모름

모름
모름

모름

모름

수지

모름

모름

모름
모름

소지섭

송지효

모름

나훈아

성룡

산다라박

모름

차은우

모름

모름
모름

옥택연

모름

모름
모름

모름
문재인

송강호

모름
김정은

모름

김현중

모름

지승현

모름

전현무

전지현

배인혁

모름

모름

김희정

모름

모름
공유

모름
소지섭

모름

모름

김종국

모름

송중기

신민아

박원순

이정재

김건희

비.

송강

유아인

모름

모름

모름

모름

박지성

모름

정우성

모름

모름

모름

모름

황인엽

모름

모름

모름

모름

모름

모름

김선영

모름

유재석

제니

모름

모름

모름
모름

이병헌

모름



In [56]:
df_with_answer = evaluate(df)

36
Total :100, 모름 : 59, 불일치 : 64, 모름 제외 불일치 : 5


In [57]:
display_dataframe_with_images(df_with_answer)


idx,image,name,answer
1,,뷔,뷔
2,,모름,장윤정
3,,김호영,설민석
4,,모름,김태리
5,,김수현,김수현
6,,모름,차준환
7,,모름,홍석천
8,,모름,김지선
9,,유관순,유관순
10,,아이유,아이유


In [23]:
df['name'] = df['name'].str.replace('\n', '', regex=False)

In [24]:
csv_file_path = "image_name_data.csv"
try:
    # 이미지 컬럼을 제외하고 저장 (이미지 데이터는 CSV에 저장하기 어려움)
    df[['name']].to_csv(csv_file_path, index=False)
    print(f"\nDataFrame saved to '{csv_file_path}'")
except Exception as e:
    print(f"\nError saving DataFrame to CSV: {e}")


DataFrame saved to 'image_name_data.csv'


## Without Grounding

In [ ]:
df_2 = pd.DataFrame(columns=['image', 'name_100%'])

for i in range(1,69):
    names_100 = detect_with_gemini_url(f"image_{i}.jpeg", 100, False)
    image_data = f"<img src=\"https://storage.googleapis.com/{BUCKET}/human_images/image_{i}.jpeg\" width=\"40%\"/>"
    new_row = pd.DataFrame({'image': [image_data], 'name_100%': [names_100]})
    df_2 = pd.concat([df_2, new_row], ignore_index=True)

성룡
윤여정
레오나르도 디카프리오
홍현희
김수현
스티브 잡스
김신영
모름
이해찬
문재인
넬슨 만델라
김혜수
패리스 힐튼
박지선
송지효
옥택연
조승우
차은우
배두나
일론 머스크
조 바이든
홍석천
시진핑
강호동
모름
유재석
김선호
유아인
전현무
제니퍼 로페즈
싸이, 손연재, 류승룡
모름
버락 오바마
고학수
전지현
유아인
워렌 버핏

저우둥위
한소희
패리스 힐튼
소지섭
벤 맥켄지
모름
김희애
모름
소지섭
조국
모름
뷔
김희철
조지 클루니
장동건
제니
모름
마이클 잭슨
강승윤
모름
손흥민
박해미
리즈 위더스푼
판빙빙
타이거 우즈
조승연
김정은
이준석
정우성
송중기, 고민시
알베르트 아인슈타인


In [146]:
display_dataframe_with_images(df_2df)

image,name_100%
,알베르트 아인슈타인


In [149]:
df_3 = pd.DataFrame(columns=['image', 'name_100%'])

for i in range(1,122):
    names_100 = detect_with_gemini_url(f"image_{i}.jpeg", 100, False)
    image_data = f"<img src=\"https://storage.googleapis.com/{BUCKET}/human_images_2/image_{i}.jpeg\" width=\"40%\"/>"
    new_row = pd.DataFrame({'image': [image_data], 'name_100%': [names_100]})
    df_3 = pd.concat([df_3, new_row], ignore_index=False)

성룡
윤여정
레오나르도 디카프리오
홍현희
김수현
스티브 잡스
김신영
모름
이해찬
문재인


KeyboardInterrupt: 